# Runnig Gemma Notebook

In [33]:
!apt-get update && apt-get install -y zstd

Hit:1 https://cli.github.com/packages stable InRelease
Hit:2 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease
Hit:3 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  InRelease
Hit:4 https://r2u.stat.illinois.edu/ubuntu jammy InRelease
Hit:5 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease
Hit:6 https://ppa.launchpadcontent.net/graphics-drivers/ppa/ubuntu jammy InRelease
Hit:7 https://ppa.launchpadcontent.net/ubuntugis/ppa/ubuntu jammy InRelease
Hit:8 http://archive.ubuntu.com/ubuntu jammy InRelease
Hit:9 http://archive.ubuntu.com/ubuntu jammy-updates InRelease
Ign:10 http://security.ubuntu.com/ubuntu jammy-security InRelease
Ign:11 http://archive.ubuntu.com/ubuntu jammy-backports InRelease
Hit:10 http://security.ubuntu.com/ubuntu jammy-security InRelease
Hit:11 http://archive.ubuntu.com/ubuntu jammy-backports InRelease
Reading package lists... Done
W: Skipping acquire of configured file 'main/source/Sources' as reposit

In [32]:
!pip install gradio -q
!pip install markdown -q

import gradio as gr
import markdown as md_lib
import pandas as pd
import plotly.express as px

from google.cloud import bigquery


In [36]:
!pip install ollama
!nohup ollama serve > /dev/null 2>&1 &

In [10]:
!apt-get update && apt-get install -y pciutils

Hit:1 https://cli.github.com/packages stable InRelease
Hit:2 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease
Hit:3 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  InRelease
Hit:4 https://r2u.stat.illinois.edu/ubuntu jammy InRelease
Hit:5 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease
Hit:6 https://ppa.launchpadcontent.net/graphics-drivers/ppa/ubuntu jammy InRelease
Hit:7 https://ppa.launchpadcontent.net/ubuntugis/ppa/ubuntu jammy InRelease
Hit:8 http://security.ubuntu.com/ubuntu jammy-security InRelease
Hit:9 http://archive.ubuntu.com/ubuntu jammy InRelease
Hit:10 http://archive.ubuntu.com/ubuntu jammy-updates InRelease
Hit:11 http://archive.ubuntu.com/ubuntu jammy-backports InRelease
Reading package lists... Done
W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)
Reading packag

In [11]:
!nvidia-smi

Tue Aug  4 15:20:10 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA L4                      Off |   00000000:00:03.0 Off |                    0 |
| N/A   40C    P8             12W /   72W |       0MiB /  23034MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [12]:
!apt-get update -qq && apt-get install -y -qq zstd
!curl -fsSL https://ollama.com/install.sh | sh

W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)
>>> Installing ollama to /usr/local
>>> Downloading ollama-linux-amd64.tar.zst
######################################################################## 100.0%
>>> Creating ollama user...
>>> Adding ollama user to video group...
>>> Adding current user to ollama group...
>>> Creating ollama systemd service...
>>> NVIDIA GPU installed.
>>> The Ollama API is now available at 127.0.0.1:11434.
>>> Install complete. Run "ollama" from the command line.


In [13]:
!pip install google-genai

In [14]:
import subprocess
process = subprocess.Popen(['ollama', 'serve'], stdout=subprocess.PIPE, stderr=subprocess.PIPE)

In [15]:
import time
time.sleep(5)
!curl http://localhost:11434

Ollama is running

In [16]:
print(process.poll())

None


In [17]:
import ollama
!ollama pull gemma3:4b

In [18]:
# import time
# time.sleep(3)
# !ollama pull moondream

In [19]:
import os
os.environ["GEMINI_API_KEY"] = "AQ.Ab8RN6IHe7AEGsu-85JCZl_gYoHNuK_Ma60InWlkwZQsvJEa6g"

In [20]:
import sys
import base64
import mimetypes
from google import genai
from google.genai import types

In [21]:
from IPython.display import display, Markdown, HTML
from google import genai
client = genai.Client(api_key=os.environ["GEMINI_API_KEY"])

In [1]:
# --- CONFIGURATION & PROMPTS ---
GEMMA_MODEL = "gemma3:4b"  # confirm exact tag with `ollama list`
GEMINI_MODEL = "gemini-3.5-flash"  # or "gemini-2.5-pro" for harder problems

CLASSIFY_PROMPT = """Look at this image. Your ONLY job is to decide: is this a high-school physics problem, diagram, or graph? Do not solve it.

CRITICAL DISQUALIFIERS (Instant "PHYSICS: no"):
- Any image featuring animals (unicorns, horses, cats, dogs, etc.), fantasy creatures, natural landscapes, portraits, or clip-art illustration WITHOUT explicit physics annotations overlaying it.
- Abstract art, general photography, textbook covers, logos, UI screenshots, or handwritten notes that do not contain a specific physics problem statement or diagram.
- Pure math or pure geometry problems without physical units, physical forces, or dynamic motion context.

ISRAELI HIGH-SCHOOL PHYSICS SCOPE:
The problem MUST belong to one of these Bagrut (5-unit) topics:
1. Mechanics: Kinematics (motion graphs, free fall, vectors), Newton's Laws (force/free-body diagrams, tension, friction, inclines), Work & Energy, Momentum & Impulse, Circular Motion (centripetal force, banked curves, vertical loops), Universal Gravitation & Kepler's Laws, Simple Harmonic Motion (pendulums, springs), Torque & Static Equilibrium.
2. Electromagnetism: Electrostatics (Coulomb's Law, field lines, potential), DC Circuits (resistors, internal resistance, EMF, meters), Magnetism (Lorentz force, right-hand rule, induction, magnetic flux).
3. Optics & Waves (if applicable): Geometric optics (refraction, Snell's law, lenses, mirrors) or wave properties (interference, diffraction, sound).

REQUIRED VISUAL SIGNALS (Must have AT LEAST ONE clear signal to say "yes"):
- Explicit physics symbols used as variables: v, a, F, m, T, ω, r, g, α, h, θ, E, P, q, B, I, R, ε, λ.
- SI unit symbols attached to numbers (in English or Hebrew/Arabic contexts): m/s, m/s², N, kg, Hz, J, W, rad/s, cm, °, V, A, Ω, C, T.
- Schematic textbook/exam diagrams: Free-body force diagrams (arrows representing F_g, N, f, T), circuit schematics (battery, resistor symbols), inclined planes, pulleys, curved tracks with labeled points (A, B, C), ray-tracing diagrams for lenses/mirrors.
- Text framing in Hebrew/Arabic/English that presents a formal physics question (e.g., wording like "גוף שמסתו", "כוח", "מהירות", "תנועה מעגלית", "מערכת צירי זמן", "חשב את").

FORMAT REQUIREMENT:
Respond in EXACTLY this format, nothing else:

PHYSICS: yes or no
DESCRIPTION: <one sentence describing what's in the image>
"""

SOLVE_PROMPT = """This image contains a physics problem.
Solve it step by step:
1. State the relevant physics concept(s)/formula(s).
2. Show the work clearly, step by step.
3. Give the final answer with correct units.
4. answer in hebrew.
"""


# --- PIPELINE FUNCTIONS ---
def load_image_b64(image_path: str):
    """Loads an image file, returning base64 string and raw bytes."""
    if not image_path or not os.path.isfile(image_path):
        raise FileNotFoundError(f"File not found at: {image_path}")

    with open(image_path, "rb") as f:
        raw_bytes = f.read()
    return base64.b64encode(raw_bytes).decode("utf-8"), raw_bytes


def gemma_classify(b64_image: str, model: str = GEMMA_MODEL) -> dict:
    """Ask Gemma whether the image is physics-related."""
    response = ollama.chat(
        model=model,
        messages=[
            {
                "role": "user",
                "images": [b64_image],
                "content": CLASSIFY_PROMPT,
            }
        ],
        keep_alive="30m",
    )
    raw_text = response["message"]["content"]

    result = {"physics": False, "description": ""}
    for line in raw_text.splitlines():
        line = line.strip()
        if line.upper().startswith("PHYSICS:"):
            result["physics"] = "yes" in line.lower()
        elif line.upper().startswith("DESCRIPTION:"):
            result["description"] = line.split(":", 1)[1].strip()

    return result


def gemini_solve(
    raw_bytes: bytes, image_path: str, gemini_model: str = GEMINI_MODEL
) -> str:
    """Send the image to Gemini to solve the physics problem."""
    client = genai.Client(api_key=os.environ["GEMINI_API_KEY"])

    mime_type, _ = mimetypes.guess_type(image_path)
    if mime_type is None:
        mime_type = "image/jpeg"

    image_part = types.Part.from_bytes(data=raw_bytes, mime_type=mime_type)

    result = client.models.generate_content(
        model=gemini_model,
        contents=[image_part, SOLVE_PROMPT],
    )
    return result.text


def student_pipeline(image) -> str:
    """Main Gradio handler: saves PIL image, runs Gemma classifier, then Gemini solver."""
    if image is None:
        return "⚠️ Please upload an image first."

    temp_path = "/content/temp_upload.png"
    image.save(temp_path)

    try:
        b64_image, raw_bytes = load_image_b64(temp_path)
    except Exception as e:
        return f"❌ Error loading image: {str(e)}"

    # Step 1: Gemma Classification
    try:
        classification = gemma_classify(b64_image)
    except Exception as e:
        return f"❌ Error running Gemma classification: {str(e)}"

    if not classification["physics"]:
        return (
            f"### ❌ Not High School Physics\n\n"
            f"**Description:** {classification['description']}\n\n"
            f"This system only processes high school (Bagrut 5-unit) physics questions."
        )

    # Step 2: Gemini Solving
    try:
        solution = gemini_solve(raw_bytes, temp_path)
    except Exception as e:
        return f"❌ Error contacting Gemini solver: {str(e)}"

    # Convert Markdown math/text to HTML for Hebrew right-to-left layout
    html_solution = md_lib.markdown(solution)
    return f"""
    <h3>✅ High School Physics Problem Detected</h3>
    <p><b>Description:</b> {classification['description']}</p>
    <hr>
    <div dir="rtl" style="text-align: right; font-size: 1.05em; line-height: 1.6;">
        {html_solution}
    </div>
    """

In [ ]:
# --- GUI BLOCK ---
with gr.Blocks(title="Israeli High School Physics Assistant") as demo:
    gr.Markdown("# ⚛️ High School Physics Problem Solver")
    gr.Markdown(
        "Upload a Bagrut 5-unit physics problem to receive a step-by-step solution in Hebrew."
    )

    with gr.Row():
        with gr.Column():
            img_input = gr.Image(type="pil", label="Upload Physics Image")
            btn_submit = gr.Button("Analyze & Solve", variant="primary")
        with gr.Column():
            solution_output = gr.HTML(label="Solution")

    btn_submit.click(
        fn=student_pipeline, inputs=[img_input], outputs=[solution_output]
    )

# Launches the web server and prints local + public (.gradio.live) URLs
demo.launch(share=True, debug=True)

In [ ]:
from google.cloud import bigquery
from google.colab import auth

auth.authenticate_user()
client = bigquery.Client(project="physics-tutoring-504312")
datasets = list(client.list_datasets())
for d in datasets:
    print(d.dataset_id)

In [ ]:
dataset_id = "physics-tutoring-504312.bagrut_data"
dataset = bigquery.Dataset(dataset_id)
dataset.location = "US"
client.create_dataset(dataset, exists_ok=True)
print("Dataset created:", dataset_id)

In [ ]:
table_id = "physics-tutoring-504312.bagrut_data.questions"

schema = [
    bigquery.SchemaField("topic", "STRING"),
    bigquery.SchemaField("description", "STRING"),
    bigquery.SchemaField("created_at", "TIMESTAMP"),
]

table = bigquery.Table(table_id, schema=schema)
table = client.create_table(table, exists_ok=True)
print("Table created:", table_id)